# Imports

In [1]:
import os
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
import optuna
import shap

import umap
import hdbscan

from joblib import dump, load
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, recall_score

/home/linkezio/Projects/A Hierarchical MEC and IoT Solution for Cyber-Attack Detection in 6G Networks/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Constants

In [2]:
RANDOM_SEED = 42

# Inputs

In [3]:
DB_PATH = "../data/sqlite/data.db"
TABLE   = "network_data"

# Samples per class (defined by the `label` column).
# - int  → same cap for every class
# - None → all available data (careful: may crash on large datasets)
SAMPLES_PER_CLASS: int | None = 10_000
# Per-class overrides — take precedence over SAMPLES_PER_CLASS.
SAMPLES_PER_CLASS_OVERRIDE: dict = {}  # e.g. {"benign": 20_000}

# Binary sampling: (n_benign, n_attack) — overrides SAMPLES_PER_CLASS when set.
# Loads n_benign rows from BENIGN_LABEL and n_attack rows from all other classes combined.
# Example: (50_000, 50_000) → 50k benign + 50k attack (pooled from all attack classes)
BINARY_SAMPLING: "tuple[int, int] | None" = None
BINARY_SAMPLING = [400000, 400000]

BENIGN_LABEL: str = "benign"

# Data

## Load Data

In [4]:
def load_dataset_from_db(
    db_path: str,
    table: str,
    samples_per_class: "int | None" = None,
    override: dict = {},
    binary_sampling: "tuple[int, int] | None" = None,
    benign_label: str = "benign",
) -> pd.DataFrame:
    """
    Load data from SQLite, sampling per class directly in the DB so that
    only the selected rows are pulled into memory.

    SQLite handles sampling with ORDER BY RANDOM() LIMIT n, avoiding the
    need to load the full dataset into pandas before filtering.

    When binary_sampling=(n_benign, n_attack) is provided, samples n_benign
    rows from benign_label and distributes n_attack evenly across all attack
    classes (n_attack // num_attack_classes per class), ignoring
    samples_per_class and override.
    """
    conn = sqlite3.connect(db_path)

    if binary_sampling is not None:
        n_benign, n_attack = binary_sampling

        q_benign = f'SELECT * FROM "{table}" WHERE label = ? ORDER BY RANDOM() LIMIT ?'
        df_benign = pd.read_sql(q_benign, conn, params=(benign_label, n_benign))
        print(f"  {benign_label}: {len(df_benign):,} rows")

        attack_classes = pd.read_sql(
            f'SELECT DISTINCT label FROM "{table}" WHERE label != ? AND label IS NOT NULL',
            conn, params=(benign_label,)
        )["label"].tolist()

        n_per_class = n_attack // len(attack_classes)
        attack_frames = []
        for cls in sorted(attack_classes):
            q_cls = f'SELECT * FROM "{table}" WHERE label = ? ORDER BY RANDOM() LIMIT ?'
            df_cls = pd.read_sql(q_cls, conn, params=(cls, n_per_class))
            attack_frames.append(df_cls)
            print(f"  {cls}: {len(df_cls):,} rows")

        conn.close()
        result = pd.concat([df_benign] + attack_frames, ignore_index=True).sample(frac=1, random_state=42)
        print(f"\nTotal: {len(result):,} rows")
        return result

    classes = pd.read_sql(
        f'SELECT DISTINCT label FROM "{table}" WHERE label IS NOT NULL', conn
    )["label"].tolist()
    print(f"Classes: {sorted(classes)}")

    frames = []
    for cls in classes:
        n = override.get(cls, samples_per_class)

        if n is None:
            query = f'SELECT * FROM "{table}" WHERE label = ?'
            df_cls = pd.read_sql(query, conn, params=(cls,))
        else:
            query = f'SELECT * FROM "{table}" WHERE label = ? ORDER BY RANDOM() LIMIT ?'
            df_cls = pd.read_sql(query, conn, params=(cls, n))

        frames.append(df_cls)
        print(f"  {cls}: {len(df_cls):,} rows")

    conn.close()

    result = pd.concat(frames, ignore_index=True).sample(frac=1, random_state=42)
    print(f"\nTotal: {len(result):,} rows")
    return result

In [5]:
def load_dataset_from_csv(
    data_root: str,
    dataset: str,
    samples_per_class: "int | None" = None,
    override: dict = {},
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Alternative: load from CSV files directly.
    Only use for small datasets — large ones will exhaust RAM.
    """
    import glob
    dataset_path = os.path.join(data_root, dataset)
    csv_paths = sorted(glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True))
    if not csv_paths:
        raise FileNotFoundError(f"No CSVs found in: {dataset_path}")

    df = pd.concat((pd.read_csv(p) for p in csv_paths), ignore_index=True)

    if samples_per_class is None and not override:
        return df

    sampled = []
    for cls in df["label"].unique():
        n = override.get(cls, samples_per_class)
        cls_df = df[df["label"] == cls]
        sampled.append(cls_df.sample(min(n, len(cls_df)), random_state=random_state) if n else cls_df)

    return pd.concat(sampled, ignore_index=True).sample(frac=1, random_state=random_state)

In [6]:
df = load_dataset_from_db(
    DB_PATH,
    TABLE,
    samples_per_class=SAMPLES_PER_CLASS,
    override=SAMPLES_PER_CLASS_OVERRIDE,
    binary_sampling=BINARY_SAMPLING,
    benign_label=BENIGN_LABEL,
)
df.head()

  benign: 400,000 rows
  bruteforce: 16,397 rows
  ddos: 50,000 rows
  dos: 50,000 rows
  malware: 50,000 rows
  mitm: 50,000 rows
  recon: 50,000 rows
  spoofing: 50,000 rows
  web: 50,000 rows

Total: 766,397 rows


,src_ip,dst_ip,src_port,dst_port,protocol,timestamp,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
320978,99.81.244.93,192.168.137.175,443,34445,6,1.665261e+09,0.115174,8.578339e+03,1.649681e+02,1.562855e+02,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
73590,157.249.81.141,192.168.137.41,443,57985,6,1.665180e+09,0.000002,7.596351e+07,1.398101e+06,1.398101e+06,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
66329,192.168.137.41,157.249.81.141,54255,80,6,1.665178e+09,0.273628,1.915009e+03,2.192758e+01,1.827298e+01,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
13908,192.168.137.41,157.249.81.141,40399,80,6,1.665185e+09,0.274235,1.910770e+03,2.187904e+01,1.823254e+01,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
533318,192.168.1.195,192.168.1.10,55842,23,6,1.749755e+09,0.009741,1.026632e+04,2.053263e+02,1.026632e+02,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


## Data Cleaning

### Removing useless features

Some columns were discarded because they did not provide useful information for training, or because they biased the information:
- Timestamp

In [7]:
removed_features: dict[str, list[str]] = {}

print("Before:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")

USELESS_FEATURES = ["timestamp"]#, "src_ip", "dst_ip"]
actually_dropped = [c for c in USELESS_FEATURES if c in df.columns]
df = df.drop(columns=actually_dropped)
removed_features["useless_features"] = actually_dropped

print("\nAfter:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")

Before:
  Shape: (766397, 83)
  Columns: ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'timestamp', 'flow_duration', 'flow_byts_s', 'flow_pkts_s', 'fwd_pkts_s', 'bwd_pkts_s', 'tot_fwd_pkts', 'tot_bwd_pkts', 'totlen_fwd_pkts', 'totlen_bwd_pkts', 'fwd_pkt_len_max', 'fwd_pkt_len_min', 'fwd_pkt_len_mean', 'fwd_pkt_len_std', 'bwd_pkt_len_max', 'bwd_pkt_len_min', 'bwd_pkt_len_mean', 'bwd_pkt_len_std', 'pkt_len_max', 'pkt_len_min', 'pkt_len_mean', 'pkt_len_std', 'pkt_len_var', 'fwd_header_len', 'bwd_header_len', 'fwd_seg_size_min', 'fwd_act_data_pkts', 'flow_iat_mean', 'flow_iat_max', 'flow_iat_min', 'flow_iat_std', 'fwd_iat_tot', 'fwd_iat_max', 'fwd_iat_min', 'fwd_iat_mean', 'fwd_iat_std', 'bwd_iat_tot', 'bwd_iat_max', 'bwd_iat_min', 'bwd_iat_mean', 'bwd_iat_std', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fin_flag_cnt', 'syn_flag_cnt', 'rst_flag_cnt', 'psh_flag_cnt', 'ack_flag_cnt', 'urg_flag_cnt', 'ece_flag_cnt', 'down_up_ratio', 'pkt_size_avg', 'fwd_se

### Removing spaces from feature names

In [8]:
spaces_before = [c for c in df.columns if c != c.strip()]
print("Before - columns with spaces:", spaces_before)

df.columns = df.columns.str.strip()

spaces_after = [c for c in df.columns if c != c.strip()]
print("After  - columns with spaces:", spaces_after)
df.head()

Before - columns with spaces: []
After  - columns with spaces: []


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
320978,99.81.244.93,192.168.137.175,443,34445,6,0.115174,8.578339e+03,1.649681e+02,1.562855e+02,8.682529,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
73590,157.249.81.141,192.168.137.41,443,57985,6,0.000002,7.596351e+07,1.398101e+06,1.398101e+06,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
66329,192.168.137.41,157.249.81.141,54255,80,6,0.273628,1.915009e+03,2.192758e+01,1.827298e+01,3.654597,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
13908,192.168.137.41,157.249.81.141,40399,80,6,0.274235,1.910770e+03,2.187904e+01,1.823254e+01,3.646507,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
533318,192.168.1.195,192.168.1.10,55842,23,6,0.009741,1.026632e+04,2.053263e+02,1.026632e+02,102.663175,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


### Removing rows with inoperative values

In [9]:
numeric_df = df.select_dtypes(include="number")

print("Before:")
print(f"  Shape: {df.shape}")
print(f"  NaN count:  {df.isna().sum().sum()}")
print(f"  Inf count:  {np.isinf(numeric_df).sum().sum()}")

df = df.replace([np.inf, -np.inf], np.nan).dropna()

numeric_df = df.select_dtypes(include="number")
print("\nAfter:")
print(f"  Shape: {df.shape}")
print(f"  NaN count:  {df.isna().sum().sum()}")
print(f"  Inf count:  {np.isinf(numeric_df).sum().sum()}")
df.head()

Before:
  Shape: (766397, 82)
  NaN count:  0
  Inf count:  0

After:
  Shape: (766397, 82)
  NaN count:  0
  Inf count:  0


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
320978,99.81.244.93,192.168.137.175,443,34445,6,0.115174,8.578339e+03,1.649681e+02,1.562855e+02,8.682529,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
73590,157.249.81.141,192.168.137.41,443,57985,6,0.000002,7.596351e+07,1.398101e+06,1.398101e+06,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
66329,192.168.137.41,157.249.81.141,54255,80,6,0.273628,1.915009e+03,2.192758e+01,1.827298e+01,3.654597,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
13908,192.168.137.41,157.249.81.141,40399,80,6,0.274235,1.910770e+03,2.187904e+01,1.823254e+01,3.646507,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
533318,192.168.1.195,192.168.1.10,55842,23,6,0.009741,1.026632e+04,2.053263e+02,1.026632e+02,102.663175,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


### Removing duplicate rows

In [10]:
print("Before:")
print(f"  Shape: {df.shape}")
print(f"  Duplicate rows: {df.duplicated().sum()}")

df = df.drop_duplicates()

print("\nAfter:")
print(f"  Shape: {df.shape}")
print(f"  Duplicate rows: {df.duplicated().sum()}")
df.head()

Before:
  Shape: (766397, 82)
  Duplicate rows: 25049

After:
  Shape: (741348, 82)
  Duplicate rows: 0


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
320978,99.81.244.93,192.168.137.175,443,34445,6,0.115174,8.578339e+03,1.649681e+02,1.562855e+02,8.682529,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
73590,157.249.81.141,192.168.137.41,443,57985,6,0.000002,7.596351e+07,1.398101e+06,1.398101e+06,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
66329,192.168.137.41,157.249.81.141,54255,80,6,0.273628,1.915009e+03,2.192758e+01,1.827298e+01,3.654597,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
13908,192.168.137.41,157.249.81.141,40399,80,6,0.274235,1.910770e+03,2.187904e+01,1.823254e+01,3.646507,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
533318,192.168.1.195,192.168.1.10,55842,23,6,0.009741,1.026632e+04,2.053263e+02,1.026632e+02,102.663175,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


### Removing highly correlated features

In [11]:
CORR_THRESHOLD = 0.95

numeric_cols = df.select_dtypes(include="number").columns.tolist()
print("Before:")
print(f"  Shape: {df.shape}")

corr_matrix = df[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]
print(f"  Dropped (corr > {CORR_THRESHOLD}): {to_drop}")

df = df.drop(columns=to_drop)
removed_features["high_correlation"] = to_drop

print("\nAfter:")
print(f"  Shape: {df.shape}")
df.head()

Before:
  Shape: (741348, 82)
  Dropped (corr > 0.95): ['bwd_pkt_len_std', 'fwd_seg_size_min', 'fwd_act_data_pkts', 'fwd_iat_tot', 'fwd_iat_max', 'psh_flag_cnt', 'ack_flag_cnt', 'pkt_size_avg', 'fwd_seg_size_avg', 'bwd_seg_size_avg', 'fwd_pkts_b_avg', 'subflow_bwd_pkts', 'subflow_bwd_byts', 'idle_max', 'idle_mean']

After:
  Shape: (741348, 67)


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,init_fwd_win_byts,init_bwd_win_byts,active_max,active_min,active_mean,active_std,idle_min,idle_std,cwr_flag_count,label
320978,99.81.244.93,192.168.137.175,443,34445,6,0.115174,8.578339e+03,1.649681e+02,1.562855e+02,8.682529,...,849,1181,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
73590,157.249.81.141,192.168.137.41,443,57985,6,0.000002,7.596351e+07,1.398101e+06,1.398101e+06,0.000000,...,508,-1,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
66329,192.168.137.41,157.249.81.141,54255,80,6,0.273628,1.915009e+03,2.192758e+01,1.827298e+01,3.654597,...,14600,65160,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
13908,192.168.137.41,157.249.81.141,40399,80,6,0.274235,1.910770e+03,2.187904e+01,1.823254e+01,3.646507,...,14600,65160,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
533318,192.168.1.195,192.168.1.10,55842,23,6,0.009741,1.026632e+04,2.053263e+02,1.026632e+02,102.663175,...,64240,5744,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


### Removing near-zero variance features

In [12]:
VARIANCE_THRESHOLD = 1e-4

numeric_cols = df.select_dtypes(include="number").columns.tolist()
print("Before:")
print(f"  Shape: {df.shape}")

variances = df[numeric_cols].var()
low_var_cols = variances[variances < VARIANCE_THRESHOLD].index.tolist()
print(f"  Dropped (var < {VARIANCE_THRESHOLD}): {low_var_cols}")

df = df.drop(columns=low_var_cols)
removed_features["near_zero_variance"] = low_var_cols

print("\nAfter:")
print(f"  Shape: {df.shape}")
df.head()

Before:
  Shape: (741348, 67)
  Dropped (var < 0.0001): ['fwd_urg_flags', 'bwd_urg_flags', 'urg_flag_cnt']

After:
  Shape: (741348, 64)


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,init_fwd_win_byts,init_bwd_win_byts,active_max,active_min,active_mean,active_std,idle_min,idle_std,cwr_flag_count,label
320978,99.81.244.93,192.168.137.175,443,34445,6,0.115174,8.578339e+03,1.649681e+02,1.562855e+02,8.682529,...,849,1181,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
73590,157.249.81.141,192.168.137.41,443,57985,6,0.000002,7.596351e+07,1.398101e+06,1.398101e+06,0.000000,...,508,-1,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
66329,192.168.137.41,157.249.81.141,54255,80,6,0.273628,1.915009e+03,2.192758e+01,1.827298e+01,3.654597,...,14600,65160,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
13908,192.168.137.41,157.249.81.141,40399,80,6,0.274235,1.910770e+03,2.187904e+01,1.823254e+01,3.646507,...,14600,65160,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
533318,192.168.1.195,192.168.1.10,55842,23,6,0.009741,1.026632e+04,2.053263e+02,1.026632e+02,102.663175,...,64240,5744,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


In [13]:
final_features = df.columns.tolist()
all_removed    = [col for cols in removed_features.values() for col in cols]

sep  = "=" * 60
sep2 = "-" * 60
lines = [
    sep,
    "FEATURE REPORT — DATA CLEANING SUMMARY",
    sep,
    "",
    f"Final features used ({len(final_features)}):",
]
for feat in sorted(final_features):
    lines.append(f"  + {feat}")

lines += ["", f"Excluded features ({len(all_removed)}):"]
for reason, cols in removed_features.items():
    if cols:
        lines += ["", f"  [{reason}]"]
        for col in sorted(cols):
            lines.append(f"    - {col}")

report = "\n".join(lines) + "\n"
print(report)

os.makedirs("../docs", exist_ok=True)
with open("../docs/features_report.txt", "w") as fh:
    fh.write(report)

print("Saved → ../docs/features_report.txt")

FEATURE REPORT — DATA CLEANING SUMMARY

Final features used (64):
  + active_max
  + active_mean
  + active_min
  + active_std
  + bwd_blk_rate_avg
  + bwd_byts_b_avg
  + bwd_header_len
  + bwd_iat_max
  + bwd_iat_mean
  + bwd_iat_min
  + bwd_iat_std
  + bwd_iat_tot
  + bwd_pkt_len_max
  + bwd_pkt_len_mean
  + bwd_pkt_len_min
  + bwd_pkts_b_avg
  + bwd_pkts_s
  + bwd_psh_flags
  + cwr_flag_count
  + down_up_ratio
  + dst_ip
  + dst_port
  + ece_flag_cnt
  + fin_flag_cnt
  + flow_byts_s
  + flow_duration
  + flow_iat_max
  + flow_iat_mean
  + flow_iat_min
  + flow_iat_std
  + flow_pkts_s
  + fwd_blk_rate_avg
  + fwd_byts_b_avg
  + fwd_header_len
  + fwd_iat_mean
  + fwd_iat_min
  + fwd_iat_std
  + fwd_pkt_len_max
  + fwd_pkt_len_mean
  + fwd_pkt_len_min
  + fwd_pkt_len_std
  + fwd_pkts_s
  + fwd_psh_flags
  + idle_min
  + idle_std
  + init_bwd_win_byts
  + init_fwd_win_byts
  + label
  + pkt_len_max
  + pkt_len_mean
  + pkt_len_min
  + pkt_len_std
  + pkt_len_var
  + protocol
  + rst_fl

# Phase 1 — Binary Classification (Benign vs Malicious)

In [14]:
y = df["label"].apply(lambda x: "benign" if x == "benign" else "attack")
X = df.drop(columns=df.select_dtypes(exclude="number").columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

A prioridade aqui é o **recall**, para minimizar falsos negativos — é preferível suspeitar de tráfego benigno do que deixar um ataque passar como benigno.

In [15]:
def objective(trial):
    threshold = trial.suggest_float("threshold", 0.01, 0.5)

    params = {
        "objective": "multiclass",
        "num_class": y_train.nunique(),
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "random_state": RANDOM_SEED,
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "max_depth": trial.suggest_int("max_depth", -1, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "max_bin": trial.suggest_int("max_bin", 128, 512),
        "class_weight": "balanced",
        "verbosity": -1,
        "force_row_wise": True,
        "n_jobs": 3,
    }

    model = lgb.LGBMClassifier(**params)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

    scores = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_val)

        attack_idx = list(model.classes_).index("attack")
        y_pred = (probs[:, attack_idx] > threshold).astype(int)
        y_val_bin = (y_val == "attack").astype(int)

        scores.append(recall_score(y_val_bin, y_pred))

    return np.mean(scores)

In [16]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, timeout=1800, show_progress_bar=True)

[I 2026-05-06 08:38:04,484] A new study created in memory with name: no-name-32cf39c6-dfad-47e0-9243-75195562a991
Best trial: 0. Best value: 1:   5%|▌         | 1/20 [01:20<25:37, 80.90s/it, 80.90/1800 seconds]

[I 2026-05-06 08:39:25,387] Trial 0 finished with value: 1.0 and parameters: {'threshold': 0.011138011329051803, 'num_leaves': 234, 'max_depth': 10, 'min_child_samples': 40, 'min_split_gain': 0.69272372800727, 'learning_rate': 0.0116893168684125, 'n_estimators': 497, 'subsample': 0.6051769282004056, 'subsample_freq': 3, 'colsample_bytree': 0.9101890082185815, 'reg_alpha': 2.014703776888729e-08, 'reg_lambda': 2.5316594258554647e-06, 'max_bin': 353}. Best is trial 0 with value: 1.0.


Best trial: 0. Best value: 1:  10%|█         | 2/20 [02:11<18:53, 62.99s/it, 131.36/1800 seconds]

[I 2026-05-06 08:40:15,840] Trial 1 finished with value: 0.8716136631330977 and parameters: {'threshold': 0.3549958121406964, 'num_leaves': 172, 'max_depth': 4, 'min_child_samples': 36, 'min_split_gain': 0.3704482701462789, 'learning_rate': 0.041515158873432, 'n_estimators': 701, 'subsample': 0.7456647759442261, 'subsample_freq': 3, 'colsample_bytree': 0.7976691460279363, 'reg_alpha': 1.9820104238311949, 'reg_lambda': 0.005744576909041631, 'max_bin': 311}. Best is trial 0 with value: 1.0.


Best trial: 0. Best value: 1:  15%|█▌        | 3/20 [03:10<17:18, 61.06s/it, 190.12/1800 seconds]

...

[I 2026-05-06 08:56:44,372] Trial 17 finished with value: 0.877356149493382 and parameters: {'threshold': 0.3109502351297513, 'num_leaves': 254, 'max_depth': 5, 'min_child_samples': 44, 'min_split_gain': 0.6469693821893322, 'learning_rate': 0.019440088512006817, 'n_estimators': 410, 'subsample': 0.6393834430872611, 'subsample_freq': 5, 'colsample_bytree': 0.8420659292150064, 'reg_alpha': 1.6264724389745327e-05, 'reg_lambda': 1.4715096988359728e-07, 'max_bin': 335}. Best is trial 0 with value: 1.0.


Best trial: 0. Best value: 1:  95%|█████████▌| 19/20 [20:31<01:15, 75.32s/it, 1231.36/1800 seconds]

[I 2026-05-06 08:58:35,846] Trial 18 finished with value: 0.9645070304390223 and parameters: {'threshold': 0.174425181359213, 'num_leaves': 113, 'max_depth': 14, 'min_child_samples': 84, 'min_split_gain': 0.4509287976301206, 'learning_rate': 0.0062044025504167846, 'n_estimators': 664, 'subsample': 0.7281369469166522, 'subsample_freq': 2, 'colsample_bytree': 0.9428114333578389, 'reg_alpha': 1.1646885901449235e-08, 'reg_lambda': 1.957516300852187e-05, 'max_bin': 243}. Best is trial 0 with value: 1.0.


Best trial: 0. Best value: 1: 100%|██████████| 20/20 [22:01<00:00, 66.06s/it, 1321.21/1800 seconds]

[I 2026-05-06 09:00:05,698] Trial 19 finished with value: 0.9947922673908561 and parameters: {'threshold': 0.08113721752581715, 'num_leaves': 233, 'max_depth': 9, 'min_child_samples': 61, 'min_split_gain': 0.01404840191690937, 'learning_rate': 0.010961131648365275, 'n_estimators': 541, 'subsample': 0.7787336551525412, 'subsample_freq': 6, 'colsample_bytree': 0.8459922542466368, 'reg_alpha': 9.89011516641911e-06, 'reg_lambda': 1.5766262490779278e-07, 'max_bin': 425}. Best is trial 0 with value: 1.0.


In [17]:
print("Best recall:", study.best_value)
print("Best params:", study.best_params)

Best recall: 1.0
Best params: {'threshold': 0.011138011329051803, 'num_leaves': 234, 'max_depth': 10, 'min_child_samples': 40, 'min_split_gain': 0.69272372800727, 'learning_rate': 0.0116893168684125, 'n_estimators': 497, 'subsample': 0.6051769282004056, 'subsample_freq': 3, 'colsample_bytree': 0.9101890082185815, 'reg_alpha': 2.014703776888729e-08, 'reg_lambda': 2.5316594258554647e-06, 'max_bin': 353}


In [18]:
best = study.best_params.copy()
threshold = best.pop("threshold")

model = lgb.LGBMClassifier(**best)

In [19]:
model.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,234
,max_depth,10
,learning_rate,0.0116893168684125
,n_estimators,497
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.69272372800727
,min_child_weight,0.001
,min_child_samples,40


In [20]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      attack       0.97      0.82      0.89     71528
      benign       0.85      0.97      0.91     76742

    accuracy                           0.90    148270
   macro avg       0.91      0.90      0.90    148270
weighted avg       0.91      0.90      0.90    148270



In [21]:
recall_score(y_test, y_pred, pos_label="benign")

0.9745510932735659

In [22]:
dump(model, "../models/binary_classifier.pkl")

['../models/binary_classifier.pkl']

# Phase 2 — Multiclassification

# Phase 3 — Clustering